# AlphaFlow — Reproducible Research Notebook

This notebook reproduces every quantitative claim in [RESEARCH.md](../RESEARCH.md)
from raw data. Run it top-to-bottom to verify the reported numbers.

**Requirements:** `pip install -r ../requirements.txt` · Python 3.11+ · Free Alpaca IEX key (optional)

**Runtime:** ~8–12 minutes (50 tickers × walk-forward LightGBM)

---

## Table of Contents

1. [Setup & Data](#1-setup)
2. [Microstructure Feature Matrix](#2-features)
3. [Walk-Forward IC & Performance](#3-walkforward)
4. [Rank Fraction Sensitivity](#4-sensitivity)
5. [Two-Tier Signal Classification](#5-classification)
6. [Benjamini-Hochberg FDR](#6-fdr)
7. [Portfolio Simulation (net-of-cost)](#7-portfolio)
8. [Daily OFI Cross-Section](#8-daily)
9. [Summary Table (RESEARCH.md §4)](#9-summary)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, t as t_dist
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

from alpha_flow.config.settings import (
    TICKERS, SIGNAL_RANK_FRACTION, SIGNAL_SIGNIFICANCE_ALPHA,
    WF_TRAIN_WINDOW, WF_TEST_WINDOW, WF_HORIZON,
)
print(f'Universe: {len(TICKERS)} tickers')
print(f'Rank fraction: {SIGNAL_RANK_FRACTION} (quintile sort — Fama-French 1993)')
print(f'FDR target Q: {SIGNAL_SIGNIFICANCE_ALPHA}')
print(f'Walk-forward: train={WF_TRAIN_WINDOW}d ({WF_TRAIN_WINDOW*5}h), test={WF_TEST_WINDOW}d ({WF_TEST_WINDOW*5}h), horizon={WF_HORIZON}')

## 1. Setup & Data <a id='1-setup'></a>

Fetch 2-year hourly bars for all 50 tickers. Uses Alpaca IEX if keys are set,
falls back to yfinance hourly.

In [ ]:
from alpha_flow.data.intraday_feed import get_intraday_bars

bars = {}
for t in TICKERS:
    df = get_intraday_bars(t, resolution='1h')
    bars[t] = df
    print(f'  {t}: {len(df)} bars  [{df.index[0].date()} → {df.index[-1].date()}]' if len(df) > 0 else f'  {t}: no data')

print(f'\nLoaded {len(bars)} tickers, median {np.median([len(v) for v in bars.values()]):.0f} bars')

## 2. Microstructure Feature Matrix <a id='2-features'></a>

Build the 13-feature matrix for one ticker (AAPL) to show feature statistics
and verify no look-ahead bias in construction.

In [ ]:
from alpha_flow.analysis.intraday_engine import build_intraday_feature_matrix, FEATURE_COLS

feats_aapl = build_intraday_feature_matrix(bars['AAPL'])
print(f'AAPL feature matrix: {feats_aapl.shape[0]} rows × {len(FEATURE_COLS)} features + target')
print(f'Features: {FEATURE_COLS}')
print(f'Date range: {feats_aapl.index[0]} → {feats_aapl.index[-1]}')
print()
display(feats_aapl[FEATURE_COLS].describe().round(4))

### Feature correlation matrix

Low inter-feature correlation means each signal contributes independent information
to the LightGBM model (desirable — highly correlated features waste splits).

In [ ]:
corr = feats_aapl[FEATURE_COLS].corr(method='spearman')
# Show pairs with |ρ| > 0.5 (potential redundancy)
high_corr = []
for i in range(len(FEATURE_COLS)):
    for j in range(i+1, len(FEATURE_COLS)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            high_corr.append((FEATURE_COLS[i], FEATURE_COLS[j], round(r, 3)))
if high_corr:
    print('Feature pairs with |Spearman ρ| > 0.5:')
    for a, b, r in high_corr:
        print(f'  {a} × {b}: {r:+.3f}')
else:
    print('No feature pairs with |ρ| > 0.5 — good orthogonality.')
print(f'\nMean off-diagonal |ρ|: {np.abs(corr.values[np.triu_indices(len(FEATURE_COLS), k=1)]).mean():.3f}')

## 3. Walk-Forward IC & Performance <a id='3-walkforward'></a>

Run the full walk-forward LightGBM pipeline on all 50 tickers.
This is the core of RESEARCH.md §4.1 — every number below should match.

In [ ]:
from alpha_flow.analysis.intraday_engine import run_intraday_pipeline

results = run_intraday_pipeline(TICKERS, resolution='1h')
print(f'Pipeline complete: {len(results)} tickers')
errors = {t: r.get('error') for t, r in results.items() if 'error' in r}
if errors:
    print(f'Errors ({len(errors)}): {errors}')

In [ ]:
# Build results table — this is Table 1 in RESEARCH.md §4.1
valid = {t: r for t, r in results.items() if 'error' not in r}

rows = []
for t, r in sorted(valid.items()):
    rows.append({
        'Ticker': t,
        'IC (%)': round(r['mean_ic'] * 100, 2),
        'IC SEM': round(r.get('ic_sem', 0) * 100, 2),
        'IC t-stat': round(r.get('ic_tstat', 0), 2),
        'IC p-value': round(r.get('ic_pvalue', 1), 4),
        'IC_IR': round(r.get('ic_ir', 0), 2),
        'Sharpe': round(r.get('sharpe', 0), 2),
        'Sortino': round(r.get('sortino', 0), 2),
        'Hit Rate': round(r.get('hit_rate', 0) * 100, 1),
        'Max DD (%)': round(r.get('max_drawdown', 0) * 100, 1),
        'Folds': r.get('n_folds', 0),
        'Latest Signal': round(r.get('latest_signal', 0), 6),
    })

df_results = pd.DataFrame(rows).set_index('Ticker')
display(df_results)

# Cross-sectional summary statistics (RESEARCH.md §4.1 headline numbers)
ics = df_results['IC (%)'].values
print(f'\n--- Cross-sectional summary (N={len(valid)} tickers) ---')
print(f'Avg |IC|:      {np.mean(np.abs(ics)):.2f}%')
print(f'Median |IC|:   {np.median(np.abs(ics)):.2f}%')
print(f'Max |IC|:      {np.max(np.abs(ics)):.2f}% ({df_results["IC (%)"].abs().idxmax()})')
print(f'Avg Sharpe:    {df_results["Sharpe"].mean():.2f}')
print(f'Avg Sortino:   {df_results["Sortino"].mean():.2f}')
print(f'Avg Hit Rate:  {df_results["Hit Rate"].mean():.1f}%')
print(f'Folds range:   {df_results["Folds"].min()}–{df_results["Folds"].max()}')

## 4. Rank Fraction Sensitivity <a id='4-sensitivity'></a>

How does the long-short book perform at different rank fractions?
This justifies `SIGNAL_RANK_FRACTION = 0.20` (quintile sort).

**What to look for:** the book Sharpe should peak or plateau around 0.15–0.25,
degrading at extremes (0.05 = too concentrated, 0.40 = too diluted).

In [ ]:
from alpha_flow.analysis.signal_classification import classify_signal

fractions = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
sensitivity_rows = []

sorted_by_signal = sorted(valid.keys(), key=lambda t: valid[t].get('latest_signal', 0), reverse=True)
n = len(sorted_by_signal)

for frac in fractions:
    n_leg = max(1, round(n * frac))
    buy_set = set(sorted_by_signal[:n_leg])
    sell_set = set(sorted_by_signal[n - n_leg:])
    
    buys, sells, holds = 0, 0, 0
    buy_ics, sell_ics = [], []
    for t in valid:
        ls = valid[t].get('latest_signal', 0)
        sig = classify_signal(
            signal_value=ls, in_buy_rank=t in buy_set, in_sell_rank=t in sell_set,
            sign_ok_buy=(ls >= 0), sign_ok_sell=(ls <= 0), abs_threshold=float('inf'),
        )
        if sig == 'BUY': buys += 1; buy_ics.append(valid[t]['mean_ic'])
        elif sig == 'SELL': sells += 1; sell_ics.append(valid[t]['mean_ic'])
        else: holds += 1
    
    avg_long_ic = np.mean(buy_ics) * 100 if buy_ics else 0
    avg_short_ic = np.mean(np.abs(sell_ics)) * 100 if sell_ics else 0
    spread_ic = avg_long_ic + avg_short_ic
    
    sensitivity_rows.append({
        'Fraction': f'{frac:.0%}',
        'Long': buys,
        'Short': sells,
        'Hold': holds,
        'Avg Long IC (%)': round(avg_long_ic, 2),
        'Avg |Short IC| (%)': round(avg_short_ic, 2),
        'IC Spread (%)': round(spread_ic, 2),
    })

df_sens = pd.DataFrame(sensitivity_rows).set_index('Fraction')
display(df_sens)
print('\nIC Spread = Avg Long IC + Avg |Short IC| — the cross-sectional edge the book monetises.')
print('Higher spread = more profitable rank sort (before transaction costs).')
print(f'Current setting: SIGNAL_RANK_FRACTION = {SIGNAL_RANK_FRACTION}')

## 5. Two-Tier Signal Classification <a id='5-classification'></a>

Demonstrate the core design: Tier-1 (tradeable book, rank-based, no FDR gate)
and Tier-2 (high-conviction flag, BH-FDR, annotation only).

In [ ]:
from alpha_flow.analysis.signal_classification import (
    classify_signal, is_high_conviction, benjamini_hochberg_threshold,
)

# Tier-1: rank by latest_signal, top/bottom 20%
n_candidates = max(1, round(n * SIGNAL_RANK_FRACTION))
buy_rank = set(sorted_by_signal[:n_candidates])
sell_rank = set(sorted_by_signal[n - n_candidates:])

# Tier-2: BH-FDR across all p-values
all_pvalues = [valid[t].get('ic_pvalue', 1.0) for t in valid]
fdr_thr = benjamini_hochberg_threshold(all_pvalues, SIGNAL_SIGNIFICANCE_ALPHA)
print(f'BH-FDR threshold at Q={SIGNAL_SIGNIFICANCE_ALPHA}: {fdr_thr:.6f}')
print(f'Best p-value in batch: {min(all_pvalues):.6f}')
print(f'Needed for rank 1 of {n}: p ≤ (1/{n})×{SIGNAL_SIGNIFICANCE_ALPHA} = {SIGNAL_SIGNIFICANCE_ALPHA/n:.6f}')
print()

classification_rows = []
for t in sorted_by_signal:
    r = valid[t]
    ls = r.get('latest_signal', 0)
    pval = r.get('ic_pvalue', 1.0)
    sig = classify_signal(
        signal_value=ls, in_buy_rank=t in buy_rank, in_sell_rank=t in sell_rank,
        sign_ok_buy=(ls >= 0), sign_ok_sell=(ls <= 0), abs_threshold=float('inf'),
    )
    hc = is_high_conviction(pval, fdr_thr)
    classification_rows.append({
        'Ticker': t, 'Latest Signal': round(ls, 6),
        'IC (%)': round(r['mean_ic'] * 100, 2),
        'p-value': round(pval, 4),
        'Tier-1 (Book)': sig,
        'Tier-2 (Conviction)': 'HIGH' if hc else '-',
    })

df_class = pd.DataFrame(classification_rows).set_index('Ticker')
display(df_class)

buys = (df_class['Tier-1 (Book)'] == 'BUY').sum()
sells = (df_class['Tier-1 (Book)'] == 'SELL').sum()
holds = (df_class['Tier-1 (Book)'] == 'HOLD').sum()
hc_count = (df_class['Tier-2 (Conviction)'] == 'HIGH').sum()
print(f'\nBook: {buys} BUY / {sells} SELL / {holds} HOLD')
print(f'High-conviction (FDR): {hc_count} of {n}')

## 6. Benjamini-Hochberg FDR — Worked Example <a id='6-fdr'></a>

Why 0 names survive FDR correction on free data:

- 50 simultaneous tests, Q = 0.10
- For the best p-value (rank 1) to survive: p₁ ≤ (1/50) × 0.10 = 0.002
- Free OHLCV hourly data produces IC ≈ 1.4% → typical p-values ≈ 0.3–0.8
- Even the best p-value (typically ≈ 0.01–0.05) exceeds 0.002

This is the **correct** statistical answer. The book still trades (Tier-1 is not gated).

In [ ]:
sorted_pvals = sorted(all_pvalues)
print(f'{"Rank":>4}  {"p-value":>10}  {"BH threshold":>14}  {"Survives?":>10}')
print('-' * 48)
for k, p in enumerate(sorted_pvals[:10], start=1):
    bh_k = (k / len(sorted_pvals)) * SIGNAL_SIGNIFICANCE_ALPHA
    survives = '  YES' if p <= bh_k else '  no'
    print(f'{k:>4}  {p:>10.6f}  {bh_k:>14.6f}  {survives:>10}')
print(f'  ...')
print(f'\nResult: {sum(1 for p in sorted_pvals if p <= fdr_thr)} names survive at Q={SIGNAL_SIGNIFICANCE_ALPHA}')

## 7. Portfolio Simulation (net-of-cost) <a id='7-portfolio'></a>

Cross-sectional long-short portfolio: long top-3 IC, short bottom-3 IC,
with Corwin-Schultz half-spread transaction costs at monthly rebalance.

In [ ]:
from alpha_flow.analysis.portfolio_engine import build_longshort_portfolio

cards = []
for t, r in valid.items():
    if r.get('equity_curve') and len(r['equity_curve']) > 10:
        cards.append({'ticker': t, **r})

port = build_longshort_portfolio(cards, n_long=3, n_short=3)

if 'error' not in port:
    print(f'Long:  {port["long_tickers"]}')
    print(f'Short: {port["short_tickers"]}')
    print(f'Gross Sharpe:    {port["gross_sharpe"]:+.4f}')
    print(f'Net Sharpe:      {port["net_sharpe"]:+.4f}')
    print(f'Avg cost (bps):  {port["avg_cost_bps"]:.1f}')
    print(f'Net Max DD:      {port["net_max_drawdown"]:.2%}')
    print(f'Hit Rate:        {port["hit_rate"]:.1%}')
    print(f'Profit Factor:   {port["profit_factor"]:.2f}')
    print(f'Rebalances:      {port["n_rebalances"]}')
    print(f'Bars:            {port["n_bars"]}')
else:
    print(f'Portfolio error: {port["error"]}')

## 8. Daily OFI Cross-Section <a id='8-daily'></a>

Daily OFI IC ≈ 0 is the expected result: Chordia et al. (2002) measured OFI
on TAQ tick data, not daily OHLCV bars. The OFI signal's half-life is ~30 min,
so daily bars average it out to noise.

In [ ]:
from alpha_flow.data.data_feed import get_daily_bars
from alpha_flow.core.ofi_calculator import rolling_ofi_zscore

daily_ics = []
for t in TICKERS:
    df = get_daily_bars(t, years=2)
    if len(df) < 50:
        continue
    ofi_z = rolling_ofi_zscore(df)
    fwd = df['close'].pct_change().shift(-1)
    common = ofi_z.dropna().index.intersection(fwd.dropna().index)
    if len(common) >= 20:
        ic, _ = spearmanr(ofi_z.loc[common], fwd.loc[common])
        if not np.isnan(ic):
            daily_ics.append((t, ic))

daily_ics_arr = np.array([x[1] for x in daily_ics])
print(f'Daily OFI IC across {len(daily_ics)} tickers:')
print(f'  Mean IC:   {np.mean(daily_ics_arr):+.4f}')
print(f'  Median IC: {np.median(daily_ics_arr):+.4f}')
print(f'  Std IC:    {np.std(daily_ics_arr):.4f}')
print(f'\nExpected: ≈ 0 on daily OHLCV (OFI half-life ~30min, daily bars average it out).')

## 9. Summary Table <a id='9-summary'></a>

This is the single source of truth. Every number in RESEARCH.md and README.md
should match this table exactly.

In [ ]:
summary = {
    'Universe size': len(TICKERS),
    'Features': len(FEATURE_COLS),
    'Walk-forward folds (range)': f'{df_results["Folds"].min()}–{df_results["Folds"].max()}',
    'Avg |IC| (%)': round(np.mean(np.abs(ics)), 2),
    'Median |IC| (%)': round(np.median(np.abs(ics)), 2),
    'Max |IC| (%)': f'{np.max(np.abs(ics)):.2f} ({df_results["IC (%)"].abs().idxmax()})',
    'Avg Sharpe': round(df_results['Sharpe'].mean(), 2),
    'Avg Sortino': round(df_results['Sortino'].mean(), 2),
    'Book (20% quintile)': f'{buys}L / {sells}S / {holds}H',
    'High-conviction (FDR)': f'{hc_count} of {n}',
    'Rank fraction': SIGNAL_RANK_FRACTION,
    'FDR target Q': SIGNAL_SIGNIFICANCE_ALPHA,
    'Daily OFI IC': f'{np.mean(daily_ics_arr):+.4f} (expected ≈ 0)',
}

for k, v in summary.items():
    print(f'{k:.<35} {v}')

print('\n--- Verify against RESEARCH.md §4 and README.md ---')
print('If any number differs, update the docs to match this notebook.')

---

## Appendix: SHAP Feature Importance (cross-sectional)

Which features matter most across all 50 tickers?

In [ ]:
# Aggregate SHAP importance across all tickers
shap_agg = {f: 0.0 for f in FEATURE_COLS}
count = 0
for t, r in valid.items():
    shap = r.get('shap_importance', {})
    if shap:
        for f in FEATURE_COLS:
            shap_agg[f] += shap.get(f, 0)
        count += 1

if count > 0:
    shap_agg = {f: v / count for f, v in shap_agg.items()}
    shap_sorted = sorted(shap_agg.items(), key=lambda x: x[1], reverse=True)
    print(f'Mean |SHAP| across {count} tickers (last fold):')
    for f, v in shap_sorted:
        bar = '█' * int(v / max(shap_agg.values()) * 30)
        print(f'  {f:<18} {v:.6f}  {bar}')